# Project: RCCI Data Analysis (University of Vaasa)
## Title: Effect of Automatic Relevancy Determination (ARD) on Engine Data
> Data Preprocessing, Data Encoding, Data Analysis, Results and Insights

In [ ]:
import os, sys
print("cwd:", os.getcwd())
print("sys.path[:5]:", sys.path[:5])

In [ ]:
%cd D:\shahnawaz\uva\main

In [ ]:
%load_ext autoreload
%autoreload 2

from src.data_loader import DataLoader
from src.data_analyzer import DataAnalyzer
from src.constant_manager import ConstantManager
from src.data_cleaner import DataCleaner
from src.feature_renamer import FeatureRenamer
from src.scaler import Scaler

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# file names declared globally for easy access
RAW_DATA_FILE = 'rcci_data_v4_5.xlsx'
PD_DATA_FILE = 'rcci_cleaned_data_v4_5.parquet'

## XLSX Raw data reading and preprocessing

In [ ]:
input_features = ['Boost pressure', 'Mass1', 'Mass2', 'Mass3', 'SOI1', 'SOI2', 'SOI3', 'H2/NG ratio', 'IVO', 'IVC', 'EVO', 'EVC']
output_features = ['IEMP', 'ITE', 'CA50', 'Lambda', 'PRR4_max', 'Pmax_CYL4', 'Nox', 'CH4', 'CO', 'NMHC', 'CO2']

loader = DataLoader(file_path=RAW_DATA_FILE)

data = loader.load_data(
    rename_columns=False,
    ignore_unnamed=True,
    skiprows=[0, 2],
    index_col=0,
    header=0
)

data.head()

In [ ]:
# data cleaning process
cleaner = DataCleaner(data)
non_empty_df = cleaner.exclude_null_columns(consider_zeros_as_null=True)

non_empty_df.head()

In [ ]:
# checking null values in the cleaned dataframe
nulls = cleaner.null_check()
print("Null values in each column:\n", nulls)

In [ ]:
encoded_df = cleaner.encode_joint_categorical_columns(col1='PARAM 1', col2='PARAM 2', new_col_name='Comb')
encoded_df.head(20)

In [ ]:
# check null values in the encoded dataframe
encoded_nulls = cleaner.null_check()
print("Null values in each column of the encoded dataframe:\n", encoded_nulls)

In [ ]:
# rename columns to match input and output features
input_features = ConstantManager().RAW_INPUT_COLUMNS
output_features = ConstantManager().RAW_OUTPUT_COLUMNS
categorical_features = ConstantManager().RAW_CATEGORICAL_COLUMNS

feature_renamer = FeatureRenamer(non_empty_df, input_features, output_features, categorical_features)

renamed_df = feature_renamer.rename_columns()
renamed_df.head()

In [ ]:
duplicates = cleaner.check_duplicates()
print("Number of duplicate rows:", duplicates)

In [ ]:
# save the cleaned and renamed dataframe to a new file
print(loader.save_df_to_parquet(renamed_df, PD_DATA_FILE))

## Read and analyze the cleaned dataframe

In [ ]:
pd_loader = DataLoader(file_path=PD_DATA_FILE)
pd_df = pd_loader.load_data()

pd_df.head()

In [ ]:
if_names, of_names, cat_names = feature_renamer.get_renamed_column_names()

print("Renamed Input Features:", if_names)

In [ ]:
import seaborn as sns

# correlation analysis of input and output features
features_corr = pd_df[if_names + of_names].corr()
plt.figure(figsize=(20, 8))
plt.title("Correlation Matrix of All Features")
sns.heatmap(features_corr, annot=True, cmap='coolwarm', fmt=".2f")
plt.show()

In [ ]:
import seaborn as sns

# correlation analysis of input features excluding if_4
input_corr = pd_df[if_names].corr()
plt.figure(figsize=(10, 8))
plt.title("Correlation Matrix of Input Features")
sns.heatmap(input_corr, annot=True, cmap='coolwarm', fmt=".2f")
plt.show()

In [ ]:
scaler_x = Scaler()
scaler_y = Scaler()

scaled_df = pd_df.copy()
scaled_df[if_names] = scaler_x.fit_transform(scaled_df, if_names)
scaled_df[of_names] = scaler_y.fit_transform(scaled_df, of_names)

In [ ]:
scaled_df.head()

In [ ]:
# find overall noise in output features
data_analyzer = DataAnalyzer(scaled_df)
output_noise_levels = data_analyzer.compute_output_noise(of_names)
print("Estimated noise levels in output features:", output_noise_levels)

# noises are in std dev units, so convert to variance for lengthscale estimation
output_noise_variances = {feature: noise**2 for feature, noise in output_noise_levels.items()}
print("Estimated noise variances in output features:", output_noise_variances)

In [ ]:
# convert dictionary of noise levels to an array in the same order as output features
output_noise_variances_array = np.array([output_noise_variances[feature] for feature in of_names])
print("Output noise levels in array form:", output_noise_variances_array)

# find true lengthscale of the data
lengthscales = data_analyzer.compute_lengthscales(if_names, of_names, noises=output_noise_variances_array)
print("Estimated lengthscales for input features:", lengthscales)

In [ ]:
# visualize lengthscales per output feature

plt.figure(figsize=(15, 12))

for i, of in enumerate(of_names):
    plt.subplot(4, 3, i+1)
    plt.bar(if_names, lengthscales[of], color='skyblue')
    plt.title(f"Estimated Lengthscales for Output Feature: {of}")
    plt.xlabel("Input Features")
    plt.ylabel("Lengthscale")
    plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# list most significant input features for each output feature based on lengthscales
# smaller lengthscale means more significant feature, so we sort in ascending order
for of in of_names:
    sorted_indices = np.argsort(lengthscales[of])
    sorted_if_names = [if_names[idx] for idx in sorted_indices]
    sorted_lengthscales = lengthscales[of][sorted_indices]
    print(f"Most significant input features for output feature '{of}':")
    for if_name, ls in zip(sorted_if_names, sorted_lengthscales):
        print(f"  {if_name}: lengthscale={ls:.4f}")

In [ ]:
# pick top 4 out of all most significant input features from all the output features
top_features = set()
for of in of_names:
    sorted_indices = np.argsort(lengthscales[of])
    top_if_names = [if_names[idx] for idx in sorted_indices[:4]]
    top_features.update(top_if_names)
print("Top input features across all output features:", top_features)

In [ ]:
print(feature_renamer.get_raw_column_name('if_4'))

In [ ]:
learned_hyperparameters = data_analyzer.construct_learned_hyperparameters(
    lengthscales, noises=output_noise_variances_array, output_columns=of_names)
learned_hyperparameters

In [ ]:
# split data into train and test sets
from sklearn.model_selection import train_test_split

training_data, testing_data = train_test_split(scaled_df, test_size=0.1, random_state=42)

print("Training data shape:", training_data.shape)
print("Testing data shape:", testing_data.shape)

In [ ]:
# Just checking output 1 to see if everything is working fine
print("Training data output feature 1 stats:")
print(training_data['of_1'].describe())

In [ ]:
fitted_gps = data_analyzer.fit_training_data(
    lengthscales=lengthscales, 
    noises = output_noise_variances_array,
    output_features=[of_names[0]], 
    X=training_data[if_names].values, 
    y=training_data[of_names].values
)

print(f"Fitted GPS: {fitted_gps}")

In [ ]:
# visualize true for output feature 1
plt.figure(figsize=(10, 6))
plt.scatter(training_data[if_names[0]], training_data[of_names[0]], alpha=0.5)
plt.xlabel("True Input (if_1)")
plt.ylabel("Value")
plt.title("True Values for Output Feature 1")
plt.show()

In [ ]:
# predict on test set using the fitted GP for output feature 1
X_test = testing_data[if_names].values
fitted_response = data_analyzer.predict(X_test, fitted_gps)
fitted_response

In [ ]:
print(testing_data[of_names[0]].values)
print(fitted_response[of_names[0]]['mean'])

In [ ]:
# plot X_test output feature 1 predictions vs true values
plt.figure(figsize=(10, 6))
plt.scatter(testing_data[if_names[0]], testing_data[of_names[0]], alpha=0.5, label='True Values')
plt.scatter(testing_data[if_names[0]], fitted_response[of_names[0]]['mean'], alpha=0.5, label='Predicted Values')
plt.xlabel("Input Feature 1 (if_1)")
plt.ylabel("Output Feature 1 (of_1)")
plt.title("True vs Predicted Values for Output Feature 1")
plt.legend()
plt.show()

In [ ]:
# plot all training data and draw confidence intervals on fitted GP predictions for output feature 1
plt.figure(figsize=(10, 6))
plt.scatter(training_data[if_names[0]], training_data[of_names[0]], alpha=0.3, label='Training Data')
plt.xlabel("Input Feature 1 (if_1)")
plt.ylabel("Output Feature 1 (of_1)")
plt.title("Training Data and Fitted GP Predictions for Output Feature 1")
# --- IGNORE ---
mean_predictions = fitted_response[of_names[0]]['mean']
stddev_predictions = fitted_response[of_names[0]]['std']
# --- IGNORE ---
plt.plot(testing_data[if_names[0]], mean_predictions, color='red', label='GP Mean Prediction')
plt.fill_between(
    testing_data[if_names[0]],
    mean_predictions - 2 * stddev_predictions,
    mean_predictions + 2 * stddev_predictions,
    color='orange',
    alpha=0.3,
    label='95% Confidence Interval'
)
plt.legend()
plt.show()